# 02 — Build retrieval index
PDFs -> chunks -> embeddings -> Qdrant + BM25
metadata -> mongomock

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import fitz
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from rank_bm25 import BM25Okapi
import mongomock

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PAPERS_DIR = DATA_DIR / "papers"

with open(DATA_DIR / "corpus_metadata.json") as f:
    corpus = json.load(f)

print(f"corpus: {len(corpus)} papers")
print(f"sample: {corpus[0]['doc_id']} - {corpus[0]['title'][:60]}")

corpus: 10 papers
sample: 2411.18583 - Automated Literature Review Using NLP Techniques and LLM-Bas


In [2]:
def parse_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc, 1):
        text = page.get_text()
        if text.strip():
            pages.append((i, text))
    doc.close()
    return pages

def chunk_text(text, size=500, overlap=80):
    out = []
    i = 0
    while i < len(text):
        c = text[i:i+size].strip()
        if c:
            out.append(c)
        i += size - overlap
    return out

all_chunks = []
for paper in tqdm(corpus, desc="chunking"):
    pdf_path = PAPERS_DIR / paper["pdf_filename"]
    if not pdf_path.exists():
        continue
    cid = 0
    for page_num, text in parse_pdf(pdf_path):
        for chunk in chunk_text(text):
            all_chunks.append({
                "chunk_id": f"{paper['doc_id']}__{cid:04d}",
                "doc_id":   paper["doc_id"],
                "page_num": page_num,
                "text":     chunk,
            })
            cid += 1

print(f"chunks: {len(all_chunks)}")
print(f"avg per paper: {len(all_chunks)/len(corpus):.1f}")

chunking:   0%|          | 0/10 [00:00<?, ?it/s]

chunks: 2118
avg per paper: 211.8


In [3]:
# bge-small downloads ~130MB on first use (one-time, cached)
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
DIM = model.get_sentence_embedding_dimension()
print(f"model loaded. dim = {DIM}")

texts = [c["text"] for c in all_chunks]
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"embeddings: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model loaded. dim = 384


/var/folders/xz/pslxf0wj2b7bzqx7kpqnfltr0000gn/T/ipykernel_47000/149925854.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/67 [00:00<?, ?it/s]

embeddings: (2118, 384)


In [4]:
qdrant = QdrantClient(":memory:")
COLLECTION = "papers"

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=emb.tolist(),
        payload={
            "chunk_id": c["chunk_id"],
            "doc_id":   c["doc_id"],
            "page_num": c["page_num"],
            "text":     c["text"],
        },
    )
    for i, (c, emb) in enumerate(zip(all_chunks, embeddings))
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"qdrant: {qdrant.count(COLLECTION).count} points")

mongo = mongomock.MongoClient()
db = mongo.papers_rag

db.papers.insert_many([dict(p) for p in corpus])
db.chunks.insert_many([{
    "_id": c["chunk_id"],
    "doc_id": c["doc_id"],
    "page_num": c["page_num"],
    "text": c["text"],
} for c in all_chunks])
print(f"mongo: papers={db.papers.count_documents({})}, chunks={db.chunks.count_documents({})}")

qdrant: 2118 points
mongo: papers=10, chunks=2118


In [5]:
tokenized = [c["text"].lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized)
print(f"bm25 indexed {len(tokenized)} chunks")

bm25 indexed 2118 chunks


In [7]:
def search_dense(q, k=5):
    qv = model.encode([q], normalize_embeddings=True)[0]
    hits = qdrant.query_points(collection_name=COLLECTION, query=qv.tolist(), limit=k).points
    return [(h.score, h.payload) for h in hits]

def search_bm25(q, k=5):
    scores = bm25.get_scores(q.lower().split())
    idx = scores.argsort()[-k:][::-1]
    return [(scores[i], all_chunks[i]) for i in idx]

q = "what is dense retrieval"

print("=== DENSE ===")
for score, p in search_dense(q):
    print(f"{score:.3f}  {p['doc_id']} p{p['page_num']}  {p['text'][:80]}...")

print("\n=== BM25 ===")
for score, c in search_bm25(q):
    print(f"{score:.3f}  {c['doc_id']} p{c['page_num']}  {c['text'][:80]}...")

=== DENSE ===
0.826  2602.07739 p1  HypRAG: Hyperbolic Dense Retrieval for Retrieval Augmented Generation
Hiren Madh...
0.799  2602.07739 p18  s.
Retrieval and RAG Inference. At inference time, dense retrieval over a corpus...
0.794  2602.07739 p9  HypRAG
Impact Statement
This paper presents work whose goal is to advance the
fi...
0.788  2602.00899 p11  ng the semantic understanding of dense retrieval with
exact matching from sparse...
0.787  2502.01113 p11  . Dense X retrieval: What retrieval granularity should we use? In Yaser
Al-Onaiz...

=== BM25 ===
9.307  2502.01113 p11  . Dense X retrieval: What retrieval granularity should we use? In Yaser
Al-Onaiz...
9.176  2502.01113 p11  tional Linguistics. doi: 10.18653/v1/2020.acl-main.117.
URL https://aclanthology...
7.976  2602.07739 p18  s.
Retrieval and RAG Inference. At inference time, dense retrieval over a corpus...
7.796  2601.05264 p23  ologies mitigate complementary deficiencies:
sparse methods excel in precise key...
7.047  2601

In [8]:
def hybrid_search(query, alpha=0.5, k=5):
    """
    Hybrid retrieval: BM25 + dense.
    alpha=0 -> pure BM25
    alpha=1 -> pure dense
    alpha=0.5 -> balanced
    
    Returns list of (score, chunk_dict) tuples, top k.
    """
    # dense retrieval
    qv = model.encode([query], normalize_embeddings=True)[0]
    dense_results = qdrant.query_points(
        collection_name=COLLECTION,
        query=qv.tolist(),
        limit=k*3,  # retrieve more for fusion
    ).points
    
    # bm25 retrieval
    bm25_scores = bm25.get_scores(query.lower().split())
    
    # score fusion - weighted sum
    chunk_scores = {}
    
    # add dense scores
    for hit in dense_results:
        cid = hit.payload["chunk_id"]
        chunk_scores[cid] = alpha * hit.score
    
    # add bm25 scores (normalized)
    max_bm25 = bm25_scores.max() if bm25_scores.max() > 0 else 1.0
    for i, score in enumerate(bm25_scores):
        cid = all_chunks[i]["chunk_id"]
        norm_score = score / max_bm25  # normalize to [0,1]
        chunk_scores[cid] = chunk_scores.get(cid, 0) + (1 - alpha) * norm_score
    
    # sort and return top k
    top_chunks = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    
    results = []
    for cid, score in top_chunks:
        # find the chunk dict
        chunk = next(c for c in all_chunks if c["chunk_id"] == cid)
        results.append((score, chunk))
    
    return results

# test it
print("Test: hybrid_search with alpha=0.5")
for score, c in hybrid_search("dense retrieval methods", alpha=0.5, k=3):
    print(f"  {score:.3f}  {c['doc_id']} p{c['page_num']}")

Test: hybrid_search with alpha=0.5
  0.765  2602.07739 p18
  0.749  2601.05264 p23
  0.726  2602.07739 p1


In [9]:
import numpy as np

# load gold set
with open(DATA_DIR / "gold_set.json") as f:
    gold_set = json.load(f)

def evaluate_retriever(search_fn, gold_set, k=5):
    """
    Compute Recall@k and NDCG@k.
    search_fn: function that takes (query, k) and returns list of (score, chunk) tuples
    """
    recalls, ndcgs = [], []
    
    for item in gold_set:
        query = item["query"]
        true_doc = item["relevant_doc"]
        
        # retrieve
        results = search_fn(query, k=k)
        retrieved_docs = [chunk["doc_id"] for _, chunk in results]
        
        # recall@k: is true_doc in top k?
        if true_doc in retrieved_docs:
            recall = 1.0
            # ndcg@k: 1 / log2(rank+1)
            rank = retrieved_docs.index(true_doc) + 1
            ndcg = 1.0 / np.log2(rank + 1)
        else:
            recall = 0.0
            ndcg = 0.0
        
        recalls.append(recall)
        ndcgs.append(ndcg)
    
    return {
        "recall@k": np.mean(recalls),
        "ndcg@k": np.mean(ndcgs),
        "n_queries": len(gold_set),
    }

# baseline: alpha=0.5, k=5
baseline_search = lambda q, k: hybrid_search(q, alpha=0.5, k=k)
baseline_metrics = evaluate_retriever(baseline_search, gold_set, k=5)

print("BASELINE (alpha=0.5, k=5):")
print(f"  Recall@5 = {baseline_metrics['recall@k']:.3f}")
print(f"  NDCG@5   = {baseline_metrics['ndcg@k']:.3f}")

BASELINE (alpha=0.5, k=5):
  Recall@5 = 1.000
  NDCG@5   = 1.000


In [10]:
import pickle

# save index objects to disk
with open(DATA_DIR / "index_objects.pkl", "wb") as f:
    pickle.dump({
        "all_chunks": all_chunks,
        "embeddings": embeddings,
    }, f)

print("Index saved to data/index_objects.pkl")

Index saved to data/index_objects.pkl


In [11]:
import numpy as np
chunk_lengths = [len(c["text"]) for c in all_chunks]
print(f"Chunk statistics:")
print(f"  Mean length: {np.mean(chunk_lengths):.0f} chars")
print(f"  Min length:  {np.min(chunk_lengths):.0f} chars")
print(f"  Max length:  {np.max(chunk_lengths):.0f} chars")

Chunk statistics:
  Mean length: 461 chars
  Min length:  4 chars
  Max length:  500 chars


In [12]:
import time
import numpy as np

latencies = []
for item in gold_set:
    start = time.time()
    results = hybrid_search(item["query"], alpha=0.5, k=5)
    latencies.append((time.time() - start) * 1000)

print(f"Baseline retrieval latency:")
print(f"  Mean:   {np.mean(latencies):.1f} ms")
print(f"  Median: {np.median(latencies):.1f} ms")
print(f"  p95:    {np.percentile(latencies, 95):.1f} ms")

Baseline retrieval latency:
  Mean:   271.9 ms
  Median: 15.6 ms
  p95:    1427.7 ms


In [2]:
# export corpus metadata as csv for D1 deliverable rubric
import pandas as pd
import json
from pathlib import Path

# define data dir explicitly in case earlier cells weren't run
DATA_DIR = Path("../data")

# load corpus metadata
with open(DATA_DIR / "corpus_metadata.json", "r") as f:
    metadata = json.load(f)

# convert to dataframe
if isinstance(metadata, dict):
    metadata = list(metadata.values())
df = pd.DataFrame(metadata)

print("columns found in metadata:")
print(df.columns.tolist())

# ensure rubric-required columns exist
required_cols = ["paper_id", "title", "authors", "venue", "year", "pdf_path", "topics"]
for col in required_cols:
    if col not in df.columns:
        df[col] = ""

# reorder
extra_cols = [c for c in df.columns if c not in required_cols]
df = df[required_cols + extra_cols]

# save
csv_path = DATA_DIR / "corpus_metadata.csv"
df.to_csv(csv_path, index=False)
print(f"\nsaved {len(df)} papers to {csv_path}")
print("\nfirst 3 rows:")
print(df.head(3))

columns found in metadata:
['doc_id', 'title', 'authors', 'abstract', 'year', 'primary_category', 'pdf_filename', 'ingested_at']

saved 10 papers to ../data/corpus_metadata.csv

first 3 rows:
  paper_id                                              title  \
0           Automated Literature Review Using NLP Techniqu...   
1           HypRAG: Hyperbolic Dense Retrieval for Retriev...   
2           Riddle Me This! Stealthy Membership Inference ...   

                                             authors venue  year pdf_path  \
0  [Nurshat Fateh Ali, Md. Mahdi Mohtasim, Shakil...        2024            
1  [Hiren Madhu, Ngoc Bui, Ali Maatouk, Leandros ...        2026            
2  [Ali Naseh, Yuefeng Peng, Anshuman Suri, Harsh...        2025            

  topics      doc_id                                           abstract  \
0         2411.18583  This research presents and compares multiple a...   
1         2602.07739  Embedding geometry plays a fundamental role in...   
2         250